## Setup

In [16]:
import pandas as pd
import requests

base_url = "https://data.elexon.co.uk/bmrs/api/v1"

START, END = "2022-06-01", "2025-06-01"

## 1. Data pulls

Pulls every dataset from the Elexon API and stores it as a `_raw` variable before any filtering, so it can be checked later (e.g. `pn_raw.head()`). 
Nothing is saved locally: variables only exist while the notebook is running, so after a restart, run all cells again. 
The full pull takes a while (19 mins total). Comment out the pulls after once variables are set.

In [ ]:
# BMU IDs: wind farm identifiers (pulled first, so BOA can be filtered to wind as it downloads)
bmunits_raw = pd.DataFrame(requests.get(f"{base_url}/reference/bmunits/all").json())
wind_ids = bmunits_raw[bmunits_raw['fuelType'] == 'WIND']['elexonBmUnit']
units = "&".join(f"bmUnit={u}" for u in wind_ids)   # url format: &bmUnit=wind_id, repeated for each unit

In [ ]:

# BOA: bid-offer acceptances and their flag
# 3 years in one request runs out of memory, so pull monthly and keep only wind farm rows from each month
# takes about 3 mins
months = pd.date_range(START, END, freq="MS")
boa_raw = pd.DataFrame()
n_all = 0   # count of all-unit acceptances, for the check in section 2
for start, end in zip(months[:-1], months[1:]):
    r = requests.get(f"{base_url}/datasets/BOALF/stream?from={start:%Y-%m-%dT00:00Z}&to={end:%Y-%m-%dT00:00Z}")
    month = pd.DataFrame(r.json())
    n_all += len(month)
    boa_raw = pd.concat([boa_raw, month[month['bmUnit'].isin(wind_ids)]], ignore_index=True)
    # print(start.date(), r.status_code, len(month))
del r, month   # free memory
boa_raw = boa_raw.drop_duplicates()   # drop repeated rows on month boundaries

In [ ]:
# PNs: physical notifications. Pulling 3 years at once returns an empty DataFrame, so pull yearly
# takes about 5 mins
years = [("2022-06-01", "2023-06-01"), ("2023-06-01", "2024-06-01"), ("2024-06-01", "2025-06-01")]
pn_raw = pd.DataFrame()
for start, end in years:
    r = requests.get(f"{base_url}/datasets/PN/stream?from={start}&to={end}&{units}")
    print(start, r.status_code, len(r.json()))   # status of pull and number of records returned
    # pn_raw = pd.concat([pn_raw, pd.DataFrame(r.json())], ignore_index=True)
pn_raw = pn_raw.drop_duplicates()   # drop repeated rows on year boundaries


In [ ]:
# Metered generation (B1610). Range limit: a year fails (error 400), so pull monthly
# takes about 5 mins
months = pd.date_range(START, END, freq="MS")   # 1st of each month
metered_gen_raw = pd.DataFrame()
for start, end in zip(months[:-1], months[1:]):   # pairs: (1 Jun, 1 Jul), (1 Jul, 1 Aug), ...
    r = requests.get(f"{base_url}/datasets/B1610/stream?settlementDateFrom={start:%Y-%m-%dT00:00Z}&settlementDateTo={end:%Y-%m-%dT00:00Z}&{units}")
    # print(start.date(), r.status_code, len(r.json()))
    metered_gen_raw = pd.concat([metered_gen_raw, pd.DataFrame(r.json())], ignore_index=True)
metered_gen_raw = metered_gen_raw.drop_duplicates()

In [ ]:
# Wind generation forecast (WINDFOR): whole period in one request
# 4 seconds
r = requests.get(f"{base_url}/datasets/WINDFOR/stream?publishDateTimeFrom={START}T00:00Z&publishDateTimeTo={END}T00:00Z")
wind_forecast_raw = pd.DataFrame(r.json())


In [ ]:
# TSDF and NDF: 3 years returns empty, so pull quarterly. TSDF has 18 boundaries (B1-B17, N): keep N (national)
# takes about 2 mins
quarters = pd.date_range(START, END, freq="3MS")
tsdf_raw, ndf_raw = pd.DataFrame(), pd.DataFrame()
for start, end in zip(quarters[:-1], quarters[1:]):
    p = f"publishDateTimeFrom={start:%Y-%m-%dT00:00Z}&publishDateTimeTo={end:%Y-%m-%dT00:00Z}"
    tsdf_raw = pd.concat([tsdf_raw, pd.DataFrame(requests.get(f"{base_url}/datasets/TSDF/stream?{p}&boundary=N").json())], ignore_index=True)
    ndf_raw = pd.concat([ndf_raw, pd.DataFrame(requests.get(f"{base_url}/datasets/NDF/stream?{p}").json())], ignore_index=True)
    # print(start.date(), len(tsdf_raw), len(ndf_raw))

In [ ]:
# INDO and ITSDO: 28 days per request
# takes about 1 min
chunks = pd.date_range(START, END, freq="28D").append(pd.DatetimeIndex([END]))
indo_raw, itsdo_raw = pd.DataFrame(), pd.DataFrame()
for start, end in zip(chunks[:-1], chunks[1:]):
    p = f"publishDateTimeFrom={start:%Y-%m-%dT00:00Z}&publishDateTimeTo={end:%Y-%m-%dT00:00Z}"
    indo_raw = pd.concat([indo_raw, pd.DataFrame(requests.get(f"{base_url}/datasets/INDO?{p}").json()['data'])], ignore_index=True)
    itsdo_raw = pd.concat([itsdo_raw, pd.DataFrame(requests.get(f"{base_url}/datasets/ITSDO?{p}").json()['data'])], ignore_index=True)

In [ ]:
# Interconnectors: 7 day limit per request (both dates inclusive)
# takes about 
ic_raw = pd.DataFrame()
for start in pd.date_range(START, "2025-05-31", freq="7D"):
    end = min(start + pd.Timedelta(days=6), pd.Timestamp("2025-05-31"))
    r = requests.get(f"{base_url}/generation/outturn/interconnectors?settlementDateFrom={start:%Y-%m-%d}&settlementDateTo={end:%Y-%m-%d}")
    ic_raw = pd.concat([ic_raw, pd.DataFrame(r.json()['data'])], ignore_index=True)

## 2. BOA: wind farm acceptances

| Column | Meaning |
|---|---|
| bmUnit | wind farm unit ID |
| settlementDate | date (UK settlement day) |
| settlementPeriodFrom / To | half-hours the instruction covers, 1-48 |
| timeFrom / timeTo | start and end of the instruction (UTC) |
| levelFrom / levelTo | level NESO instructed (MW) |
| acceptanceNumber | ID of the instruction |
| acceptanceTime | when NESO issued the instruction |
| soFlag | True = constraint action |

In [25]:
# keep only the columns needed to build curtailment
cols = ['bmUnit', 'settlementDate', 'settlementPeriodFrom', 'settlementPeriodTo',
        'timeFrom', 'timeTo', 'levelFrom', 'levelTo',
        'acceptanceNumber', 'acceptanceTime', 'soFlag']
boa = boa_raw[cols].copy()

# convert text times to datetimes (the Z means UTC)
for c in ['timeFrom', 'timeTo', 'acceptanceTime']:
    boa[c] = pd.to_datetime(boa[c], utc=True)
boa['settlementDate'] = pd.to_datetime(boa['settlementDate'])

# already filtered to wind farms during the pull
boa_wind = boa

print("all acceptances:", n_all)
print("wind acceptances:", boa_wind.shape)
print("wind units registered:", wind_ids.nunique())
print("wind units with at least one acceptance:", boa_wind['bmUnit'].nunique())

all acceptances: 8099549
wind acceptances: (1242003, 11)
wind units registered: 241
wind units with at least one acceptance: 192


## 3. PN: physical notifications

| Column | Meaning |
|---|---|
| bmUnit | wind farm unit ID |
| settlementDate | date (UK settlement day) |
| settlementPeriod | half-hour the plan covers, 1-48 |
| timeFrom / timeTo | start and end of the half-hour (UTC) |
| levelFrom / levelTo | level the unit planned (MW) |

In [26]:
# remove unnecessary columns, convert time columns to datetime format as done with boa dataset
pn = pn_raw.drop(columns=['dataset', 'nationalGridBmUnit'])
for c in ['timeFrom', 'timeTo']:
    pn[c] = pd.to_datetime(pn[c], utc=True)
pn['settlementDate'] = pd.to_datetime(pn['settlementDate'])

print(pn.shape)

(11978896, 7)


## 4. Calculate curtailment

Curtailment = (PN − accepted level) × time, where PN is higher (Storey et al.).

NESO can change accepted levels within a time interval. The latest one replaces the previous, so where a farm's acceptances overlap, the earlier one is cut where the next one starts.

In [27]:
# all of each wind farm's acceptances, in time order (acceptanceNumber breaks ties: later acceptance second)
boa_sorted = boa_wind.sort_values(['bmUnit', 'timeFrom', 'acceptanceNumber']).copy()

# start time of the next acceptance for the same farm
next_start = boa_sorted.groupby('bmUnit')['timeFrom'].shift(-1)

# overlap: next acceptance starts before this one ends
cut = next_start < boa_sorted['timeTo']
print("overlapping acceptances:", cut.sum())

# how far along the old acceptance had gone before the new acceptance interrupted
fraction = (next_start - boa_sorted['timeFrom']) / (boa_sorted['timeTo'] - boa_sorted['timeFrom'])

# level at the cut (calculated levels have decimals, so change level cols from integer to float)
boa_sorted[['levelFrom', 'levelTo']] = boa_sorted[['levelFrom', 'levelTo']].astype(float)
boa_sorted.loc[cut, 'levelTo'] = boa_sorted['levelFrom'] + (boa_sorted['levelTo'] - boa_sorted['levelFrom']) * fraction

# end the earlier acceptance where the next one starts
boa_sorted.loc[cut, 'timeTo'] = next_start

overlapping acceptances: 598556


In [28]:
# 1. how long each instruction lasted, and its average level
boa_sorted['hours'] = (boa_sorted['timeTo'] - boa_sorted['timeFrom']).dt.total_seconds() / 3600
boa_sorted['boaLevel'] = (boa_sorted['levelFrom'] + boa_sorted['levelTo']) / 2

# 2. planned level (PN) for each farm and half-hour
pn['pnLevel'] = (pn['levelFrom'] + pn['levelTo']) / 2
pn_sp = pn.groupby(['bmUnit', 'settlementDate', 'settlementPeriod'], as_index=False)['pnLevel'].mean()

# merge PN level onto accepted levels (inner: keeps acceptances that have a matching PN)
curtailment = boa_sorted.merge(pn_sp, left_on=['bmUnit', 'settlementDate', 'settlementPeriodFrom'],
                               right_on=['bmUnit', 'settlementDate', 'settlementPeriod'], how='inner')

# 3. curtailed MWh = (planned - instructed) x hours, only where planned is higher --> clip lower=0
curtailment['curtailed_MWh'] = (curtailment['pnLevel'] - curtailment['boaLevel']).clip(lower=0) * curtailment['hours']

# 4. hour the instruction began in (UTC)
curtailment['hour'] = curtailment['timeFrom'].dt.floor('h')

n_rows = len(curtailment)   # every feature merge below should keep this number of rows
print("acceptances with a matching PN:", n_rows, "of", len(boa_sorted))
print("wind units after PN merge:", curtailment['bmUnit'].nunique())
print("acceptances with curtailment > 0 (SO / non-SO):")
print(curtailment[curtailment['curtailed_MWh'] > 0]['soFlag'].value_counts())

acceptances with a matching PN: 1235082 of 1242003
wind units after PN merge: 192
acceptances with curtailment > 0 (SO / non-SO):
soFlag
True     695734
False     98029
Name: count, dtype: int64


## 5. Features

Day-ahead framing: forecasts must be published before the target day; actual values (metered generation, INDO, ITSDO, interconnectors) are only known after the fact, so they are lagged one day.

### Metered generation (B1610), per wind farm, lagged one day

In [29]:
metered_gen = metered_gen_raw[['bmUnit', 'settlementDate', 'settlementPeriod', 'settlementRunType', 'quantity']].copy()
metered_gen['settlementDate'] = pd.to_datetime(metered_gen['settlementDate'])

# data is refreshed by later settlement runs: keep the latest run for each unit and half-hour
print(metered_gen['settlementRunType'].unique())   # check all runs are in run_order
run_order = {'II': 0, 'SF': 1, 'R1': 2, 'R2': 3, 'R3': 4, 'RF': 5, 'DF': 6}
metered_gen['run'] = metered_gen['settlementRunType'].map(run_order)
metered_gen = metered_gen.sort_values('run').drop_duplicates(['bmUnit', 'settlementDate', 'settlementPeriod'], keep='last')

# lag one day: each row gets the previous day's metered generation for the same half-hour
metered_gen['settlementDate'] = metered_gen['settlementDate'] + pd.Timedelta(days=1)
metered_gen = metered_gen[['bmUnit', 'settlementDate', 'settlementPeriod', 'quantity']].rename(columns={'quantity': 'metered_MWh_lag1d'})

curtailment = curtailment.merge(metered_gen, on=['bmUnit', 'settlementDate', 'settlementPeriod'], how='left')
print(len(curtailment) == n_rows, curtailment['metered_MWh_lag1d'].isna().sum())

<ArrowStringArray>
['DF', 'RF']
Length: 2, dtype: str
True 455


### Wind generation forecast (WINDFOR): GB total, same value for every wind farm in the same hour

In [30]:
wind_forecast = wind_forecast_raw.copy()
wind_forecast['publishTime'] = pd.to_datetime(wind_forecast['publishTime'], utc=True)
wind_forecast['startTime'] = pd.to_datetime(wind_forecast['startTime'], utc=True)

# day-ahead: keep the latest forecast published at least 24 hours before each startTime
wind_forecast = wind_forecast[wind_forecast['startTime'] - wind_forecast['publishTime'] >= pd.Timedelta(hours=24)]
wind_forecast = wind_forecast.sort_values('publishTime').drop_duplicates('startTime', keep='last')

# forecast is already hourly: rename to match curtailment
wind_forecast = wind_forecast.rename(columns={'startTime': 'hour', 'generation': 'wind_forecast_MW'})

curtailment = curtailment.merge(wind_forecast[['hour', 'wind_forecast_MW']], on='hour', how='left')
print(len(curtailment) == n_rows, curtailment['wind_forecast_MW'].isna().sum())

True 0


### TSDF and NDF: day-ahead demand forecasts

In [31]:
keys = ['settlementDate', 'settlementPeriod']

for raw, name in [(tsdf_raw, 'tsdf_MW'), (ndf_raw, 'ndf_MW')]:
    df = raw.copy()
    df['publishTime'] = pd.to_datetime(df['publishTime'], utc=True)
    df['startTime'] = pd.to_datetime(df['startTime'], utc=True)

    # day-ahead: latest forecast published before the target day starts
    # e.g. a forecast for 15:30 on 3 June is kept if published on 2 June or earlier, dropped if published on 3 June
    df = df[df['publishTime'] < df['startTime'].dt.floor('D')]
    df = df.sort_values('publishTime').drop_duplicates('startTime', keep='last')

    df['settlementDate'] = pd.to_datetime(df['settlementDate'])
    curtailment = curtailment.merge(df[keys + ['demand']].rename(columns={'demand': name}), on=keys, how='left')

print(len(curtailment) == n_rows, curtailment[['tsdf_MW', 'ndf_MW']].isna().sum().tolist())

True [0, 0]


### INDO, ITSDO and interconnectors: actual values, lagged one day

Missing lagged values fall on clock-change days: October days have 50 periods (49 and 50 have nothing to lag from), and the day after the March change follows a 46-period day.

In [32]:
def lag_1day(df, value, name):
    df = df.copy()
    df['settlementDate'] = pd.to_datetime(df['settlementDate']) + pd.Timedelta(days=1)
    return df[keys + [value]].rename(columns={value: name})

indo = indo_raw.drop_duplicates(keys)
itsdo = itsdo_raw.drop_duplicates(keys)

# interconnectors: total flow across all interconnectors per settlement period
ic = ic_raw.copy()
ic_total = ic.groupby(keys, as_index=False)['generation'].sum()

for lagged in [lag_1day(indo, 'demand', 'indo_MW_lag1d'),
               lag_1day(itsdo, 'demand', 'itsdo_MW_lag1d'),
               lag_1day(ic_total, 'generation', 'interconnector_MW_lag1d')]:
    curtailment = curtailment.merge(lagged, on=keys, how='left')

lag_cols = ['indo_MW_lag1d', 'itsdo_MW_lag1d', 'interconnector_MW_lag1d']
print(len(curtailment) == n_rows, curtailment[lag_cols].isna().sum().tolist())
print(curtailment.loc[curtailment['indo_MW_lag1d'].isna(), 'settlementDate'].value_counts())

True [455, 455, 477]
settlementDate
2023-10-29    186
2024-10-27    152
2022-10-30     63
2024-04-01     44
2023-03-27     10
Name: count, dtype: int64


## 6. Save

The only file saved: the final dataset (shared in Releases).

| Column | Meaning |
|---|---|
| bmUnit, settlementDate, settlementPeriodFrom/To, timeFrom/timeTo, acceptanceNumber, acceptanceTime, soFlag | same as BOA |
| levelFrom / levelTo | level NESO instructed (MW), with earlier instructions cut where a later one starts |
| hours | how long the instruction lasted |
| boaLevel | average instructed level over the instruction (MW) |
| settlementPeriod | the PN half-hour matched to the instruction |
| pnLevel | average planned level for that half-hour (MW) |
| curtailed_MWh | (pnLevel − boaLevel) × hours, 0 where planned is not higher |
| hour | start of the hour the instruction began in (UTC) |
| metered_MWh_lag1d | the farm's metered generation, same half-hour previous day |
| wind_forecast_MW | GB wind forecast for that hour, published ≥24h before |
| tsdf_MW, ndf_MW | day-ahead demand forecasts for that half-hour |
| indo_MW_lag1d, itsdo_MW_lag1d | actual demand, same half-hour previous day |
| interconnector_MW_lag1d | total interconnector flow, same half-hour previous day |

In [ ]:
print(curtailment.shape)
curtailment.head()

(1235082, 24)


,bmUnit,settlementDate,settlementPeriodFrom,settlementPeriodTo,timeFrom,timeTo,levelFrom,levelTo,acceptanceNumber,acceptanceTime,...,pnLevel,curtailed_MWh,hour,metered_MWh_lag1d,wind_forecast_MW,tsdf_MW,ndf_MW,indo_MW_lag1d,itsdo_MW_lag1d,interconnector_MW_lag1d
0,2__BLOND003,2025-04-16,28,28,2025-04-16 12:48:00+00:00,2025-04-16 12:49:00+00:00,16.0,2.0,1,2025-04-16 12:47:00+00:00,...,16.0,0.116667,2025-04-16 12:00:00+00:00,0.045,13272,24758,23222,28598.0,29372.0,6724.0
1,2__BLOND003,2025-04-16,28,29,2025-04-16 12:49:00+00:00,2025-04-16 13:19:00+00:00,0.0,0.0,1,2025-04-16 12:47:00+00:00,...,16.0,8.000000,2025-04-16 12:00:00+00:00,0.045,13272,24758,23222,28598.0,29372.0,6724.0
2,2__BLOND003,2025-04-16,29,29,2025-04-16 13:19:00+00:00,2025-04-16 13:20:00+00:00,0.0,1.0,1,2025-04-16 12:47:00+00:00,...,16.0,0.258333,2025-04-16 13:00:00+00:00,-0.034,13250,25664,22629,28297.0,29177.0,6762.0
3,2__BLOND003,2025-05-25,23,23,2025-05-25 10:14:00+00:00,2025-05-25 10:15:00+00:00,10.0,2.0,702,2025-05-25 10:13:00+00:00,...,10.0,0.066667,2025-05-25 10:00:00+00:00,3.861,18744,17920,12695,20994.0,24060.0,-76.0
4,2__BLOND003,2025-05-25,23,24,2025-05-25 10:15:00+00:00,2025-05-25 10:45:00+00:00,0.0,0.0,702,2025-05-25 10:13:00+00:00,...,10.0,5.000000,2025-05-25 10:00:00+00:00,3.861,18744,17920,12695,20994.0,24060.0,-76.0
